In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path

CLEAN_DATA_DIR = Path("../clean_data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEAN_DATA_DIR}/4_eea_co2_emissions_from_passenger_cars-001.parquet")
df.show(5)

In [ ]:
from pyspark.sql import functions as F

num_cols = [
    "mass_in_running_order (kg)", "co2_emissions_WLTP (g/km)", 
    "engine_capacity (cm3)", "engine_power (KW)", "electric_energy_consumption (Wh/km)"
]
str_cols = ["member_state", "make", "commercial_name", "fuel_type", "fuel_mode"]
all_cols = str_cols + num_cols

def count_missing(col_name):
    is_missing = F.col(col_name).isNull() | (F.trim(F.col(col_name)) == "")
    return F.sum(is_missing.cast("int")).alias(f"{col_name}_nulls")

def count_zeros(col_name):
    is_zero = F.col(col_name) == 0
    return F.sum(is_zero.cast("int")).alias(f"{col_name}_zeros")

zeros = [count_missing(c) for c in all_cols] + [count_zeros(c) for c in num_cols]

df_zeros = df.groupBy("year").agg(*zeros).orderBy("year")
df_zeros.show(truncate=False)

In [ ]:
before = df.count()
# clean the dataset, str cols cannot be na
df = df.na.drop(subset=str_cols)
# these columns introduce verbosity and so are not needed
not_needed = ["vehicle_family_id_number", "version"]
df = df.drop(*not_needed)
df.select("year").distinct().show(), print(f"dropped {before - df.count()} rows, {df.count()} rows left")

In [ ]:
# there are many fuel types entered in many different ways, we have to merge them
df.select("fuel_type").distinct().count(), df.select("fuel_type").distinct().show(100)

In [ ]:
df = df.withColumn("fuel_type", F.lower(F.col("fuel_type")))
df.select("fuel_type").distinct().count(), df.select("fuel_type").distinct().show(100)

In [ ]:
# some values have spaces around them, we don't want those to count as distinct types
df = df.withColumn("fuel_type", F.trim(F.col("fuel_type")))
# some values use dashes and end up duplicating existing value types
df = df.withColumn("fuel_type", F.regexp_replace(F.col("fuel_type"), "-", "/"))
df.select("fuel_type").distinct().count(), df.select("fuel_type").distinct().show(100)

In [ ]:
unnecessary = ["unknown", "other"]
df = df.filter(~df.fuel_type.isin(unnecessary))
df.select("fuel_type").distinct().count(), df.select("fuel_type").distinct().show(100), print(f"{df.count()} rows left")

In [ ]:
# aggregate
df = df.withColumn(
    "fuel_type",
    F.when(F.col("fuel_type") == "electric", "Electric")
     .when(F.col("fuel_type") == "petrol phev", "Plug-in Hybrid")
     .when(F.col("fuel_type").isin("petrol/electric", "diesel/electric", "hybrid/petrol/e"), "Hybrid")
     .when(F.col("fuel_type") == "petrol", "Petrol")
     .when(F.col("fuel_type") == "diesel", "Diesel")
     .otherwise("Alternative/Other") 
)

In [ ]:
df.count(), df.show(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

df_aggregated = (
    df.groupBy("member_state", "year")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy("member_state", "year")
)

pdf = df_aggregated.toPandas()

pivot_df = pdf.pivot(
    index="member_state", columns="year", values="row_count"
)

pivot_df_k = pivot_df / 1000.0

plt.figure(figsize=(12, 10))
sns.set_theme(style="white")

ax = sns.heatmap(
    pivot_df_k,
    annot=True,
    fmt=".1f",
    cbar_kws={"label": "Record Count (in Thousands)"},
    linewidths=0.5,
    linecolor="white",
)

plt.title(
    "Country-Level Record Completeness & Registration Volume (2014-2023)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Registration Year", fontsize=12, labelpad=10)
plt.ylabel("Member State (ISO2 Code)", fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

target_col = "mass_in_running_order (kg)"
num_bins = 500

stats = (
    df.filter(F.col(target_col).isNotNull())
    .select(F.min(target_col).alias("min_val"), F.max(target_col).alias("max_val"))
    .collect()[0]
)

min_val, max_val = int(stats["min_val"]), int(stats["max_val"])
bin_width = (max_val - min_val) / num_bins

df_binned = (
    df.filter(F.col(target_col).isNotNull())
    .withColumn(
        "bin_center",
        F.floor((F.col(target_col) - min_val) / bin_width) * bin_width
        + min_val
        + (bin_width / 2),
    )
)

hist_counts = (
    df_binned.groupBy("fuel_type", "bin_center")
    .agg(F.count("*").alias("count"))
    .orderBy("bin_center")
)

pdf_hist = hist_counts.toPandas()

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

sns.lineplot(
    data=pdf_hist,
    x="bin_center",
    y="count",
    hue="fuel_type",
    linewidth=2.5,
)

for fuel in pdf_hist["fuel_type"].unique():
    subset = pdf_hist[pdf_hist["fuel_type"] == fuel]
    plt.fill_between(subset["bin_center"], subset["count"], alpha=0.25)

plt.title(
    f"Distribution of {target_col}",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Mass in Running Order (kg)", fontsize=12, labelpad=10)
plt.ylabel("Vehicle Registration Count", fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

In [ ]:
# heaviest vehicles
(
    df.filter(F.col("mass_in_running_order (kg)") > 3500)
    .select("make", "commercial_name", "fuel_type", "mass_in_running_order (kg)")
    .orderBy(F.desc("mass_in_running_order (kg)"))
    .show(20)
)